# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install -q mlcroissant

## 1. Data Loading
Load Croissant metadata and prepare for loading actual data records.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Set the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")
print(f"License: {metadata.license}")

## 2. Data Overview
List available record sets, their fields, and `@id`s, referencing each by `@id`.

Croissant datasets organize tabular and semi-structured data as 'record sets'; fields correspond to columns or attributes. We will fetch the available record sets, their field lists, and the corresponding column IDs.

In [ ]:
# List all record sets by their @id
record_sets = getattr(metadata, 'recordSet', [])

print(f"Found {len(record_sets)} record set(s).\n")
record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or rs
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    # Each record set has fields; display them
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        # f can be a string (@id) or an object
        fid = getattr(f, '@id', None) or f
        print(f"  Field @id: {fid}")

Let's inspect the first few records of each record set (by `@id`).

Below, for each record set, we print the first record as an example. This gives you field names and an idea of the data shape.

In [ ]:
for record_set_id in record_set_ids:
    print(f"\nFirst record from RecordSet @id={record_set_id}:")
    try:
        records = dataset.records(record_set=record_set_id)
        for i, rec in enumerate(records):
            print(rec)
            if i >= 0:  # print only the first
                break
    except Exception as e:
        print(f"  Could not load records: {e}")

## 3. Data Extraction
Load tabular data from each record set into a pandas DataFrame for analysis.

We use the `@id` of each record set. Refer to the printed IDs above.

In [ ]:
# Prepare a DataFrame for each record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet @id={record_set_id}\nFields: {df.columns.tolist()}")
            print(df.head(2))
    except Exception as e:
        print(f"Could not load DataFrame for {record_set_id}: {e}")

For the following analysis, we'll focus on the main protein record set (replace below with actual record set `@id` found in the previous output).

In [ ]:
# Pick the main record set. If only one exists, use it.
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"Working DataFrame columns: {df.columns.tolist()}")
    df.head()

## 4. Exploratory Data Analysis (EDA)
We will demonstrate numeric filtering, normalization and, if possible, grouping. You can change `numeric_field_id` and `group_field_id` based on the column names shown above.

In [ ]:
# --- Set field @id or column name for numeric and group field (adjust as needed/available) ---
df = dataframes[main_record_set_id]

# Try common candidates for numeric analysis
candidates = ['coverage_percent', 'num_peptides', 'MW', 'abundance', 'normalized_abundance', 'pI']
found = False
for cand in candidates:
    if cand in df.columns:
        numeric_field = cand
        found = True
        break
if not found:
    # Use the first numeric column if above not present
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.empty else None

print(f"Numeric field for analysis: {numeric_field}")

# Filter out low values
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Add normalized column
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field (e.g., 'sample'/'description')
group_field_candidates = ['description', 'accession', 'sample', 'modification']
group_field = None
for gf in group_field_candidates:
    if gf in df.columns:
        group_field = gf
        break
if group_field:
    print(f"\nGrouping by '{group_field}' (showing group means):")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field using a histogram and, if a group field was found, a boxplot by group.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Histogram
plt.figure(figsize=(6, 4))
df[numeric_field].hist(bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field is available, boxplot
if group_field:
    plt.figure(figsize=(8, 4))
    filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=90)
    plt.title(f"{numeric_field} by {group_field}")
    plt.suptitle("")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and explored a FAIR^2 Croissant-formatted mass spectrometry dataset.
- Used `mlcroissant` to inspect available record sets, fields (by `@id`), and data sample records.
- Extracted tabular data to pandas DataFrames, performed filtering, normalization, and grouping analyses.
- Visualized feature distributions to support further downstream use.

You can adapt this workflow to any dataset provided in Croissant format by referencing entities by their `@id` and using the data programmatically with `mlcroissant`.